In [ ]:
%%capture

import altair as alt
import gcsfs
import pandas as pd

#from calitp_data_analysis import calitp_color_palette as cp
from IPython.display import HTML, Markdown, display
#from update_vars import GCS_FILE_PATH, MONTH, PUBLIC_FILENAME, YEAR
#from _01_ntd_ridership_utils import sum_by_group
from gtfs_curator_utils import magics

alt.data_transformers.enable("vegafusion")

WIDTH = 300
HEIGHT = 150

In [ ]:
# parameters cell for local
rtpa = "Metropolitan Transportation Commission"

In [ ]:
%%capture_parameters
rtpa

# {rtpa}
Annual Ridership Trends

Download data from our **[public folder](https://console.cloud.google.com/storage/browser/calitp-publish-data-analysis)** by navigating to `ntd_annual_ridership` and selecting a file.

Transit operators/agencies that submit annual reports to NTD are included in this report. Reporters that were previously active reporters, but are currently not, may appear. This may result in Reporters showing zero or partial ridership data in the report.

If a Reporter, type of service, mode, or any combination of, is not a annual reporter or has not reported data since 2018, they will not appear in the report.

Examples:

* **Reporter A** is an annual reporter from 2019-2022, then became inactive and did not report for 2023. Reporter A's ridership data will be displayed for 2019-2022 only.
* **Reporter B** is an annual from 2000-2017, then became inactive and did not report for 2018. Reporter B will be named in the report, but will not display ridership data.
* **Reporter C** was an inactive reporter form 2015-2020, then became an active full reporter for 2021. Reporter C's ridership data will be displayed for 2021-present.


# need to set PUBLIC_FILENAME in update_vars
URL = "https://console.cloud.google.com/storage/" "browser/calitp-publish-data-analysis"

display(
    HTML(
        f"""
        <a href={URL}>
        Download the latest month of data: {PUBLIC_FILENAME}</a>
        """
    )
)

In [ ]:
# read in data
GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/ntd_explore/"

crosswalk = pd.read_parquet(
    f"{GCS_FILE_PATH}crosswalk.parquet", 
    filesystem=gcsfs.GCSFileSystem(),
    filters = [[("rtpa_name", "==", rtpa)]]
).rename(columns = {"ntd_id_2022": "ntd_id"})

#full_df = pd.read_parquet(f"{GCS_FILE_PATH}annual.parquet",filesystem=gcsfs.GCSFileSystem())

df = pd.read_parquet(
    f"{GCS_FILE_PATH}annual.parquet",
    filesystem=gcsfs.GCSFileSystem(),
).merge(
    crosswalk,
    on = "ntd_id",
    how = "left"
)

In [ ]:
initial_agg = df.groupby("ntd_id").agg(
    total_upt=("unlinked_passenger_trips", "sum")
).reset_index()


In [ ]:
import B3_ntd_utils as ntd_utils

In [ ]:
#by_agency_long = 

In [ ]:
def group_by_agency(df):
    """
    Take in the 'by_ageny_long' df and aggregatese by rtpa, and calculates upt % of total.
    To be used in pie chart
    """
    initial_agg = df.groupby("source_agency").agg(total_upt=("unlinked_passenger_trips", "sum")).reset_index()

    # % total columns
    initial_agg["pct_of_total_upt"] = (
        initial_agg["total_upt"] / initial_agg["total_upt"].sum()
    ) * 100

    # cleaning data types and rounding
    initial_agg["total_upt"] = initial_agg["total_upt"].astype("int64")
    initial_agg["pct_of_total_upt"] = initial_agg["pct_of_total_upt"].round(decimals=2)
    cleaned_agg = initial_agg.sort_values(by="total_upt", ascending=False)

    return cleaned_agg